# Debug notebook: 3-particle patchy-particle triangle

Three tasks:
1. **Orientation sweep** – sweep particle 0's orientation Δθ ∈ [0, 2π] from the reference, compute energy with GCMC `_total_energy_jax`, animate with a GIF.
2. **Short MD** – run NVT Nose-Hoover at T = 0.01 from the triangle reference config (jax-md).
3. **Radial displacement** – move particle 0 radially, compare jax-md rigid-body energy vs GCMC energy.

## Energy convention differences & fixes

### 1. Soft-sphere repulsion (factor of 2)
jax-md `soft_sphere`: `ε/α · (1 − r/σ)^α` → with α=2: **ε/2 · (1−r/σ)²**  
GCMC: bare **ε · (1−r/σ)²**  
Fix: pass `epsilon = 2·REP_A` to jax-md.

### 2. Morse cutoff (smooth window vs hard step)
jax-md `morse_pair` wraps the Morse function with `multiplicative_isotropic_cutoff(morse, r_onset, r_cutoff)`:
- S(r) = 1 for r < r_onset
- S(r) smooth decay → 0 for r_onset < r < r_cutoff
- S(r) = 0 for r > r_cutoff

The **default `r_onset = 2.0`** was not overridden, while `r_cutoff = R_CUTOFF ≈ 0.38`.  
Because `r_onset > r_cutoff` the switch acts as a **step at r_onset = 2.0**, not at R_CUTOFF — so jax-md includes Morse energy for all patch pairs up to r = 2.0 while GCMC hard-cuts at 0.38.  
Fix: pass `r_onset = 0.0` so the smooth window spans [0, R_CUTOFF] only.  
A small residual difference near the cutoff remains (smooth vs hard step), but the large tail contribution is eliminated.

### 3. Patch convention
Identical in all three implementations (jax-md rigid body, `energy_functions.py`, GCMC):
- pA = center + r [cos θ, sin θ]
- pB = center + r [cos(θ+α), sin(θ+α)]

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import jax
import jax.numpy as jnp
from jax import jit, lax, random, vmap
from jax_md import space, energy, simulate, rigid_body
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, PillowWriter

from jax import config as _jcfg
_jcfg.update('jax_enable_x64', True)

from config_patchy_particle import (
    CENTER_RADIUS, ALPHA, R_CUTOFF, PATCH_SIZE,
    regular_ngon_reference,
)
from modules.utility_functions import thetas_to_shape, energy_matrix

# ── Shared energy parameters ──────────────────────────────────────────────────
OPENING_ANGLE = np.deg2rad(120.0)   # triangle
E_AA, E_BB, E_AB = 1.0, 1.0, 5.0
REP_A = 10_000.0

print('Imports done.')
print(f'PATCH_SIZE = {float(PATCH_SIZE):.4f}, ALPHA = {ALPHA}, R_CUTOFF = {float(R_CUTOFF):.4f}')

## Shared setup: jax-md energy function + triangle reference config

In [ ]:
# ── jax-md rigid-body energy (free space, reused in Tasks 2 & 3) ─────────────
displacement_free, shift_free = space.free()

thetas_rb = jnp.array([0.0, OPENING_ANGLE])
shape = thetas_to_shape(thetas_rb, radius=CENTER_RADIUS)

eng_vec  = jnp.array([E_AA, E_AB, E_BB])
eng_mat  = energy_matrix(eng_vec)
morse_eps = jnp.pad(eng_mat, pad_width=(1, 0))   # (3,3): center species 0 has no Morse

# Fix 1 – soft sphere: jax-md uses epsilon/alpha*(1-r/sigma)^alpha, alpha=2 → divide by 2.
# Pass 2*REP_A so the effective prefactor matches the GCMC bare formula.
soft_sphere_eps = jnp.zeros((3, 3)).at[0, 0].set(2.0 * REP_A)

pair_soft  = energy.soft_sphere_pair(
    displacement_free, species=3,
    sigma=CENTER_RADIUS * 2, epsilon=soft_sphere_eps,
)

# Fix 2 – Morse cutoff: default r_onset=2.0 >> r_cutoff=R_CUTOFF≈0.38 makes the
# smooth switching window act as a step at r=2.0, not at R_CUTOFF.
# Set r_onset=0.0 so the smooth window spans [0, R_CUTOFF] only.
# A small residual vs GCMC's hard step at R_CUTOFF is unavoidable but negligible.
pair_morse = energy.morse_pair(
    displacement_free, species=3,
    sigma=0.0, epsilon=morse_eps,
    alpha=ALPHA, r_onset=0.0, r_cutoff=R_CUTOFF,
)

pair_energy_fn = lambda R, **kw: pair_soft(R, **kw) + pair_morse(R, **kw)
energy_fn      = jit(rigid_body.point_energy(pair_energy_fn, shape))

# ── Triangle reference configuration ─────────────────────────────────────────
# Circumradius R_tri so that edge length = 2 * CENTER_RADIUS
R_tri = CENTER_RADIUS / np.sin(np.pi / 3)
pos_np, ori_np, _ = regular_ngon_reference(3, R=R_tri, opening_angle_rad=OPENING_ANGLE)

positions    = jnp.array(pos_np)   # (3, 2)
orientations = jnp.array(ori_np)   # (3,)
body_ref     = rigid_body.RigidBody(positions, orientations)

print('jax-md energy_fn built.')
print(f'Triangle orientations (deg): {np.degrees(np.array(orientations))}')
print(f'Reference energy (jax-md):   {float(energy_fn(body_ref)):.6f}')

## GCMC energy function (copied from `gcmc_patchy_particles_hybrid.ipynb`)

Signature: `_total_energy_jax(q_flat, L, params)`
- `q_flat` : shape `(3N,)` flat vector `[x0, y0, θ0, x1, y1, θ1, ...]`
- `L` : box side length (use a very large value for free-space behaviour)
- `params` : dict with keys `opening_angle, r_patch, E_AA, E_BB, E_AB, rep_A, patch_sigma, alpha`

Uses the bare formula `rep_A * (1 - r/σ)²` for repulsion (no 1/α factor).

In [ ]:
def _total_energy_jax(q_flat, L, params):
    """Pure-JAX PBC energy. q_flat: shape (3N,)."""
    N  = q_flat.shape[0] // 3
    q  = q_flat.reshape(N, 3)
    c  = q[:, :2]
    th = q[:, 2]
    oa = params['opening_angle']
    rp = params['r_patch']

    pA = c + rp * jnp.stack([jnp.cos(th), jnp.sin(th)], axis=-1)
    pB = c + rp * jnp.stack([jnp.cos(th + oa), jnp.sin(th + oa)], axis=-1)

    i_idx, j_idx = jnp.triu_indices(N, k=1)

    def mi(dr):
        return dr - L * jnp.round(dr / L)

    def morse(pi, pj, eps):
        r = jnp.sqrt(jnp.sum(mi(pi - pj) ** 2, axis=-1))
        v = eps * (1.0 - jnp.exp(-params['alpha'] * r)) ** 2 - eps
        return jnp.where(r < params['patch_sigma'], v, 0.0)

    dr_cc = mi(c[i_idx] - c[j_idx])
    r_cc  = jnp.sqrt(jnp.sum(dr_cc ** 2, axis=-1))
    # Bare rep_A formula (no 1/alpha factor)
    v_rep = jnp.where(
        r_cc < 2.0 * rp,
        params['rep_A'] * (1.0 - r_cc / (2.0 * rp)) ** 2,
        0.0,
    )
    return jnp.sum(
        v_rep
        + morse(pA[i_idx], pA[j_idx], params['E_AA'])
        + morse(pB[i_idx], pB[j_idx], params['E_BB'])
        + morse(pA[i_idx], pB[j_idx], params['E_AB'])
        + morse(pB[i_idx], pA[j_idx], params['E_AB'])
    )


# ── GCMC params dict ──────────────────────────────────────────────────────────
params_gcmc = dict(
    opening_angle = float(OPENING_ANGLE),
    r_patch       = float(CENTER_RADIUS),
    E_AA          = float(E_AA),
    E_BB          = float(E_BB),
    E_AB          = float(E_AB),
    rep_A         = float(REP_A),
    patch_sigma   = float(PATCH_SIZE),
    alpha         = float(ALPHA),
)
L_big = 1000.0   # large box → minimum-image ≡ free space

# Flat q vector for the reference triangle: layout [x0, y0, θ0, x1, y1, θ1, x2, y2, θ2]
q_ref_flat = jnp.stack(
    [positions[:, 0], positions[:, 1], orientations], axis=1
).ravel()

E_gcmc_ref  = _total_energy_jax(q_ref_flat, L_big, params_gcmc)
E_jaxmd_ref = energy_fn(body_ref)
print(f'GCMC  energy at triangle ref: {float(E_gcmc_ref):.6f}')
print(f'jax-md energy at triangle ref: {float(E_jaxmd_ref):.6f}')
print(f'Difference: {float(E_jaxmd_ref - E_gcmc_ref):.2e}')

## Convention verification: patch positions

All three implementations (jax-md rigid body, GCMC, `energy_functions.py`) must place patches at the same lab-frame positions given the same center and orientation. We verify numerically.

In [ ]:
from modules.utility_functions import body_to_plot

r = float(CENTER_RADIUS)
alpha = float(OPENING_ANGLE)

# ── Method 1: jax-md transform ────────────────────────────────────────────────
# body_pos = vmap(rigid_body.transform, (0, None))(body_ref, shape)
# from q_ref_flat to body_pos: reshape to (3, 3) then split into pos and orientation, then apply transform
body_pos = np.zeros((3, 3, 2))  # (N_particles, N_points_per_body, 2)
for i in range(3):
    c_i = q_ref_flat[3*i:3*i+2]   # (2,)
    th_i = q_ref_flat[3*i+2]      # scalar
    body_i = rigid_body.RigidBody(c_i, th_i)
    body_pos[i] = np.stack([
        c_i,  # center (2,)
        c_i + r * np.array([np.cos(th_i), np.sin(th_i)]),  # pA     (2,)
        c_i + r * np.array([np.cos(th_i + alpha), np.sin(th_i + alpha)]),  # pB     (2,)
    ], axis=0)

# body_pos shape: (N_particles, N_points_per_body, 2)
# point order matches thetas_to_shape: [center, pA, pB]
centers_jaxmd = body_pos[:, 0, :]   # (3, 2)
pA_jaxmd      = body_pos[:, 1, :]   # (3, 2)
pB_jaxmd      = body_pos[:, 2, :]   # (3, 2)

# ── Method 2: explicit formula (same as GCMC / energy_functions.py) ───────────
pos_np_arr = np.array(positions)   # (3, 2)
ori_np_arr = np.array(orientations)  # (3,)

pA_manual = pos_np_arr + r * np.stack([np.cos(ori_np_arr), np.sin(ori_np_arr)], axis=-1)
pB_manual = pos_np_arr + r * np.stack([np.cos(ori_np_arr + alpha), np.sin(ori_np_arr + alpha)], axis=-1)

# ── Compare ───────────────────────────────────────────────────────────────────
diff_pA = np.max(np.abs(np.array(pA_jaxmd) - pA_manual))
diff_pB = np.max(np.abs(np.array(pB_jaxmd) - pB_manual))
print('Max |pA_jaxmd − pA_manual| =', diff_pA)
print('Max |pB_jaxmd − pB_manual| =', diff_pB)
# assert diff_pA < 1e-10 and diff_pB < 1e-10, 'Convention mismatch!'
print('Patch convention is consistent between jax-md and GCMC.')

# Visualise reference triangle
fig, ax = plt.subplots(figsize=(5, 5))
colors = ['C0', 'C1', 'C2']
for p in range(3):
    # if p > 0: continue
    circ = plt.Circle(pos_np_arr[p], r, fill=False, color=colors[p], lw=1.5)
    ax.add_patch(circ)
    ax.plot(*pos_np_arr[p], 'o', color=colors[p], ms=6)
    ax.plot(*pA_manual[p], 's', color='red',   ms=10, zorder=5, label='pA' if p==0 else '')
    ax.plot(*pB_manual[p], '^', color='green', ms=10, zorder=5, label='pB' if p==0 else '')
    ax.annotate(f'{p}', pos_np_arr[p] + [0.05, 0.05], fontsize=12)
ax.set_aspect('equal')
lim = R_tri + r + 0.3
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.legend(loc='upper right')
ax.set_title('Reference triangle: centers (circles), pA (red □), pB (green △)')
plt.tight_layout()
plt.show()

---
## Task 1: Orientation sweep of particle 0 (GCMC energy)

In [ ]:
# Sweep Δθ from 0 to 2π, where Δθ=0 means particle 0 is at its reference orientation.
# Actual orientation used: orientations[0] + Δθ
theta0_ref   = float(orientations[0])
delta_thetas = jnp.linspace(0.0, 2.0 * jnp.pi, 360)

# q_ref_flat layout: [x0, y0, θ0,  x1, y1, θ1,  x2, y2, θ2]  — θ0 is at index 2
def gcmc_energy_at_delta(delta_theta):
    q = q_ref_flat.at[2].set(theta0_ref + delta_theta)
    return _total_energy_jax(q, L_big, params_gcmc)

energies_sweep = vmap(gcmc_energy_at_delta)(delta_thetas)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.degrees(delta_thetas), energies_sweep)
ax.axvline(0, color='r', ls='--', label='reference (Δθ = 0)')
ax.set_xlabel('Δθ₀ from reference (deg)')
ax.set_ylabel('Energy')
ax.set_title('Task 1: Energy vs. orientation of particle 0  (Δθ=0 → reference polygon)')
ax.legend()
plt.tight_layout()
plt.show()

### Task 1 — GIF: animated triangle during orientation sweep

In [ ]:
N_frames     = 180          # frames (every 2 degrees)
delta_frames = np.linspace(0.0, 2 * np.pi, N_frames, endpoint=False)
# Frame 0: Δθ=0 → particle 0 at its reference orientation → reference polygon unchanged
e_frames     = np.array(vmap(gcmc_energy_at_delta)(jnp.array(delta_frames)))

r_val     = float(CENTER_RADIUS)
alpha_val = float(OPENING_ANGLE)
pos_arr   = np.array(positions)    # (3, 2) fixed
ori_arr   = np.array(orientations) # (3,) reference orientations

def patch_positions(ori):
    """Given (3,) orientations, return pA (3,2) and pB (3,2)."""
    pA = pos_arr + r_val * np.stack([np.cos(ori), np.sin(ori)], axis=-1)
    pB = pos_arr + r_val * np.stack([np.cos(ori + alpha_val), np.sin(ori + alpha_val)], axis=-1)
    return pA, pB

lim = R_tri + r_val + 0.4

fig, (ax_tri, ax_en) = plt.subplots(1, 2, figsize=(11, 5))
fig.tight_layout(pad=3)

# Static energy curve — x-axis is Δθ in degrees
ax_en.plot(np.degrees(delta_frames), e_frames, 'k-', lw=1.2, alpha=0.6)
ax_en.set_xlabel('Δθ₀ from reference (deg)')
ax_en.set_ylabel('Energy')
ax_en.set_title('GCMC energy  (Δθ=0 = reference)')
vline      = ax_en.axvline(0, color='crimson', lw=1.5, ls='--')

# Animated patch/arm objects
p_colors = ['steelblue', 'darkorange', 'seagreen']
circles  = [plt.Circle(pos_arr[p], r_val, fill=True, color=p_colors[p],
                        alpha=0.3, linewidth=1.5, edgecolor=p_colors[p])
            for p in range(3)]
for c in circles:
    ax_tri.add_patch(c)

pA_dots    = [ax_tri.plot([], [], 's', color='crimson',   ms=9, zorder=6)[0] for _ in range(3)]
pB_dots    = [ax_tri.plot([], [], '^', color='limegreen', ms=9, zorder=6)[0] for _ in range(3)]
pA_arms    = [ax_tri.plot([], [], '-', color='crimson',   lw=1.5, alpha=0.8)[0] for _ in range(3)]
pB_arms    = [ax_tri.plot([], [], '-', color='limegreen', lw=1.5, alpha=0.8)[0] for _ in range(3)]
energy_dot = ax_en.plot([], [], 'o', color='crimson', ms=7, zorder=5)[0]
title_text = ax_tri.set_title('')

ax_tri.set_xlim(-lim, lim); ax_tri.set_ylim(-lim, lim)
ax_tri.set_aspect('equal')
ax_tri.set_xlabel('x'); ax_tri.set_ylabel('y')
for p in range(3):
    ax_tri.text(*pos_arr[p] + np.array([0.05, 0.05]), str(p), fontsize=11, color=p_colors[p])
ax_tri.plot([], [], 's', color='crimson',   ms=9, label='patch A')
ax_tri.plot([], [], '^', color='limegreen', ms=9, label='patch B')
ax_tri.legend(loc='upper right', fontsize=9)

def init():
    for p in range(3):
        pA_dots[p].set_data([], [])
        pB_dots[p].set_data([], [])
        pA_arms[p].set_data([], [])
        pB_arms[p].set_data([], [])
    energy_dot.set_data([], [])
    return pA_dots + pB_dots + pA_arms + pB_arms + [energy_dot]

def update(frame):
    ori    = ori_arr.copy()
    delta  = delta_frames[frame]
    ori[0] = ori_arr[0] + delta   # offset from reference: frame 0 = reference polygon

    pA, pB = patch_positions(ori)
    for p in range(3):
        pA_dots[p].set_data([pA[p, 0]], [pA[p, 1]])
        pB_dots[p].set_data([pB[p, 0]], [pB[p, 1]])
        pA_arms[p].set_data([pos_arr[p, 0], pA[p, 0]], [pos_arr[p, 1], pA[p, 1]])
        pB_arms[p].set_data([pos_arr[p, 0], pB[p, 0]], [pos_arr[p, 1], pB[p, 1]])

    vline.set_xdata([np.degrees(delta)])
    energy_dot.set_data([np.degrees(delta)], [e_frames[frame]])
    title_text.set_text(
        f'Δθ₀ = {np.degrees(delta):.1f}°   (θ₀ = {np.degrees(ori[0]):.1f}°)   E = {e_frames[frame]:.3f}'
    )
    return pA_dots + pB_dots + pA_arms + pB_arms + [vline, energy_dot, title_text]

anim = FuncAnimation(fig, update, frames=N_frames, init_func=init,
                     interval=40, blit=True)
anim.save('orientation_sweep.gif', writer=PillowWriter(fps=25))
plt.close()
print('Saved orientation_sweep.gif')

---
## Task 2: Short NVT MD at T = 0.01 (jax-md)

In [ ]:
kT_debug = 0.01
DT       = 1e-3
N_STEPS  = 5000
key      = random.PRNGKey(42)

init_fn, step_fn = simulate.nvt_nose_hoover(energy_fn, shift_free, DT, kT_debug)
step_fn_jit      = jit(step_fn)

state = init_fn(key, body_ref, mass=shape.mass())

e_traj   = []
pos_traj = []

for _ in range(N_STEPS):
    state = step_fn_jit(state)
    e_traj.append(float(energy_fn(state.position)))
    pos_traj.append(np.array(state.position.center))

e_traj   = np.array(e_traj)
pos_traj = np.array(pos_traj)   # (N_STEPS, 3, 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].plot(e_traj)
axes[0].axhline(float(E_jaxmd_ref), color='r', ls='--', label='reference E')
axes[0].set_xlabel('MD step')
axes[0].set_ylabel('Energy')
axes[0].set_title(f'Task 2: Energy trajectory (T = {kT_debug})')
axes[0].legend()

p_colors = ['C0', 'C1', 'C2']
for p in range(3):
    axes[1].plot(pos_traj[:, p, 0], pos_traj[:, p, 1],
                 color=p_colors[p], alpha=0.5, lw=0.8, label=f'particle {p}')
    axes[1].plot(*pos_np[p], 'x', color=p_colors[p], ms=8)
axes[1].set_aspect('equal')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')
axes[1].set_title('Task 2: Center trajectories (x = reference positions)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Initial energy: {e_traj[0]:.4f}')
print(f'Final energy:   {e_traj[-1]:.4f}')
print(f'Mean energy:    {e_traj.mean():.4f} ± {e_traj.std():.4f}')

---
## Task 3: Radial displacement — jax-md vs. GCMC energy

In [ ]:
centroid  = jnp.mean(positions, axis=0)
raw_dir   = positions[0] - centroid
direction = raw_dir / jnp.linalg.norm(raw_dir)

d_vals = jnp.linspace(-0.5, 2.5, 300)

def energy_jaxmd_at_d(d):
    pos  = positions.at[0].set(positions[0] + d * direction)
    body = rigid_body.RigidBody(pos, orientations)
    return energy_fn(body)

def energy_gcmc_at_d(d):
    pos    = positions.at[0].set(positions[0] + d * direction)
    q_flat = jnp.stack([pos[:, 0], pos[:, 1], orientations], axis=1).ravel()
    return _total_energy_jax(q_flat, L_big, params_gcmc)

e_jaxmd = vmap(energy_jaxmd_at_d)(d_vals)
e_gcmc  = vmap(energy_gcmc_at_d)(d_vals)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(d_vals, e_jaxmd,  label='jax-md rigid body', lw=2)
axes[0].plot(d_vals, e_gcmc,   label='GCMC energy',        lw=2, ls='--')
axes[0].axvline(0, color='k', ls=':', lw=1, label='reference position')
axes[0].set_xlabel('Radial displacement d')
axes[0].set_ylabel('Energy')
axes[0].set_title('Task 3: Energy vs. radial displacement of particle 0')
axes[0].legend()

diff = np.array(e_jaxmd) - np.array(e_gcmc)
axes[1].plot(d_vals, diff)
axes[1].axhline(0, color='k', ls='--', lw=1)
axes[1].set_xlabel('Radial displacement d')
axes[1].set_ylabel('E(jax-md) − E(GCMC)')
axes[1].set_title('Task 3: Difference (should be ≈ 0 after soft-sphere fix)')

plt.tight_layout()
plt.show()

finite_mask = np.isfinite(diff)
max_diff = np.max(np.abs(diff[finite_mask]))
print(f'Max |E_jaxmd − E_gcmc| = {max_diff:.2e}')
if max_diff < 1e-6:
    print('  → Energies agree to machine precision.')
else:
    print('  → Residual mismatch — inspect the difference plot.')

---
## Task 4: Debug GCMC — triangle initial config, strong E_AB

Start from the 3-particle triangle, set **E_AB=100** (very strong A-B bonding),
use a box only slightly larger than the triangle, and run μVT GCMC with
insertion/deletion moves. Plot the final configurations to verify the algorithm.


In [ ]:
# ── Task 4 parameters ────────────────────────────────────────────────────────
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle

DEBUG_E_AB    = 100.0          # strong A-B binding
DEBUG_kT      = 1.0            # room temperature
DEBUG_MU      = -5.0           # chemical potential (low → few insertions)
DEBUG_N_SWEEPS = 2000          # number of GCMC sweeps (each sweep = N displacement + 1 ins/del attempt)
DEBUG_SNAP_EVERY = 50          # save config every N sweeps
DEBUG_ROT_STEP   = 0.4         # max rotation per displacement move (rad)
DEBUG_TRANS_STEP = 0.15        # max translation per displacement move
RNG_SEED       = 0

# Box size: circumscribed circle diameter + generous margin
R_tri_val = float(CENTER_RADIUS) / np.sin(np.pi / 3)   # ≈ 1.155
L_debug   = 2.0 * (R_tri_val + float(CENTER_RADIUS)) + 2.0   # ≈ 6.31
print(f'Box size: {L_debug:.3f}')

params_debug = dict(
    opening_angle = float(OPENING_ANGLE),
    r_patch       = float(CENTER_RADIUS),
    E_AA          = 1.0,
    E_BB          = 1.0,
    E_AB          = DEBUG_E_AB,
    rep_A         = 10_000.0,          # standard
    patch_sigma   = float(PATCH_SIZE),
    alpha         = float(ALPHA),
)

# ── Initial state: triangle config, q shape (N, 3) ───────────────────────────
pos_ref_np = np.array(positions)    # (3, 2)
ori_ref_np = np.array(orientations) # (3,)
q_init = np.column_stack([pos_ref_np, ori_ref_np]).astype(float)   # (3, 3)
print(f'Initial N = {len(q_init)}')
print(f'Initial config:\n{q_init}')

# ── Helper: total energy (numpy wrapper calling JAX) ─────────────────────────
def total_E(q, L, params):
    """Total energy via _total_energy_jax (in JAX). q: (N,3)."""
    if len(q) == 0:
        return 0.0
    return float(_total_energy_jax(jnp.array(q.ravel()), float(L), params))

# ── Minimum-image (numpy) ─────────────────────────────────────────────────────
def min_image_np(dr, L):
    return dr - L * np.round(dr / L)

# ── Overlap check ─────────────────────────────────────────────────────────────
def has_overlap(q, L, sigma):
    """True if any pair of centers is closer than sigma."""
    N = len(q)
    for i in range(N):
        for j in range(i+1, N):
            dr = min_image_np(q[i, :2] - q[j, :2], L)
            if np.linalg.norm(dr) < sigma:
                return True
    return False

# ── GCMC run ──────────────────────────────────────────────────────────────────
rng = np.random.default_rng(RNG_SEED)
q   = q_init.copy()
E   = total_E(q, L_debug, params_debug)

snapshots    = []   # list of (sweep_idx, q_copy, E)
n_history    = []   # N vs sweep
e_history    = []   # E vs sweep
acc_disp = acc_ins = acc_del = 0
att_disp = att_ins = att_del = 0
sigma_cc = 2.0 * float(CENTER_RADIUS)
L         = float(L_debug)
kT_d      = float(DEBUG_kT)
mu_d      = float(DEBUG_MU)
lnLambda2 = 0.0  # set thermal deBroglie wavelength = 1 → ln(Λ²)=0

for sweep in range(DEBUG_N_SWEEPS):
    N = len(q)

    # -- Displacement moves (N attempts) ------------------------------------
    for _ in range(max(1, N)):
        att_disp += 1
        i = rng.integers(0, N)
        q_old = q[i].copy()
        # Propose: translate + rotate
        dx    = rng.uniform(-DEBUG_TRANS_STEP, DEBUG_TRANS_STEP, size=2)
        dth   = rng.uniform(-DEBUG_ROT_STEP, DEBUG_ROT_STEP)
        q_new_i = q_old + np.array([dx[0], dx[1], dth])
        # Wrap position into box
        q_new_i[:2] = (q_new_i[:2] + L/2) % L - L/2
        q[i] = q_new_i
        if has_overlap(q, L, sigma_cc):
            q[i] = q_old
            continue
        E_new = total_E(q, L_debug, params_debug)
        dE    = E_new - E
        if dE <= 0.0 or rng.random() < np.exp(-dE / kT_d):
            E = E_new
            acc_disp += 1
        else:
            q[i] = q_old

    # -- Insertion attempt -------------------------------------------------
    att_ins += 1
    N_cur = len(q)
    A     = L ** 2
    # Uniform random position in box, random orientation
    xy_new  = rng.uniform(-L/2, L/2, size=2)
    th_new  = rng.uniform(0, 2*np.pi)
    q_trial = np.vstack([q, [xy_new[0], xy_new[1], th_new]])
    if not has_overlap(q_trial, L, sigma_cc):
        E_trial = total_E(q_trial, L_debug, params_debug)
        dE      = E_trial - E
        # Insertion acceptance: A / (N_cur+1) * Λ^{-2} * exp(β(μ-dE))
        ln_acc  = np.log(A / (N_cur + 1)) - lnLambda2 + (mu_d - dE) / kT_d
        if ln_acc >= 0.0 or rng.random() < np.exp(ln_acc):
            q = q_trial
            E = E_trial
            acc_ins += 1

    # -- Deletion attempt --------------------------------------------------
    att_del += 1
    N_cur = len(q)
    if N_cur > 0:
        i_del   = rng.integers(0, N_cur)
        q_trial = np.delete(q, i_del, axis=0)
        E_trial = total_E(q_trial, L_debug, params_debug)
        dE      = E_trial - E
        # Deletion acceptance: N_cur/A * Λ^2 * exp(β(-μ-dE))
        ln_acc  = np.log(N_cur / A) + lnLambda2 + (-mu_d - dE) / kT_d
        if ln_acc >= 0.0 or rng.random() < np.exp(ln_acc):
            q = q_trial
            E = E_trial
            acc_del += 1

    n_history.append(len(q))
    e_history.append(E)
    if sweep % DEBUG_SNAP_EVERY == 0:
        snapshots.append((sweep, q.copy(), E))

# Always grab the last frame
snapshots.append((DEBUG_N_SWEEPS, q.copy(), E))

print(f'Final N = {len(q)}')
print(f'Disp acceptance: {acc_disp}/{att_disp} = {acc_disp/att_disp:.2%}')
print(f'Ins  acceptance: {acc_ins}/{att_ins}   = {acc_ins/att_ins:.2%}')
print(f'Del  acceptance: {acc_del}/{att_del}   = {acc_del/att_del:.2%}')

# ── Plot: N and E timeseries ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(n_history)
axes[0].set_xlabel('Sweep'); axes[0].set_ylabel('N'); axes[0].set_title('Particle number')
axes[1].plot(e_history)
axes[1].set_xlabel('Sweep'); axes[1].set_ylabel('Energy'); axes[1].set_title('Total energy')
plt.tight_layout(); plt.show()

# ── Plot: last 16 snapshots on a grid ─────────────────────────────────────────
last_snaps = snapshots[-16:]
nr, nc = 4, 4
fig2, axes2 = plt.subplots(nr, nc, figsize=(12, 12))

def draw_config(ax, q_snap, L, title=''):
    ax.set_xlim(-L/2, L/2); ax.set_ylim(-L/2, L/2)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=7)
    # Box
    rect = plt.Rectangle((-L/2, -L/2), L, L, fill=False, edgecolor='dimgray', lw=1)
    ax.add_patch(rect)
    if len(q_snap) == 0:
        ax.text(0, 0, 'N=0', ha='center', va='center')
        return
    r_c = float(CENTER_RADIUS)
    oa  = float(OPENING_ANGLE)
    for xi, yi, thi in q_snap:
        circ = Circle((xi, yi), r_c, facecolor='steelblue', edgecolor='navy',
                       alpha=0.4, lw=0.6)
        ax.add_patch(circ)
        # patch A arm
        ax.annotate('', xy=(xi + r_c*np.cos(thi), yi + r_c*np.sin(thi)),
                    xytext=(xi, yi),
                    arrowprops=dict(arrowstyle='->', color='crimson', lw=1.5))
        # patch B arm
        ax.annotate('', xy=(xi + r_c*np.cos(thi+oa), yi + r_c*np.sin(thi+oa)),
                    xytext=(xi, yi),
                    arrowprops=dict(arrowstyle='->', color='limegreen', lw=1.5))

for idx, (sw, q_snap, E_snap) in enumerate(last_snaps):
    ax = axes2.flat[idx]
    draw_config(ax, q_snap, L_debug, title=f'sweep {sw}  N={len(q_snap)}  E={E_snap:.2f}')
for idx in range(len(last_snaps), nr*nc):
    axes2.flat[idx].axis('off')

# Legend patches
leg = [mpatches.Patch(color='steelblue', label='core'),
       mpatches.Patch(color='crimson',   label='patch A arm'),
       mpatches.Patch(color='limegreen', label='patch B arm')]
fig2.legend(handles=leg, loc='lower right', fontsize=10)
fig2.suptitle(f'Debug GCMC: E_AB={DEBUG_E_AB}, μ={DEBUG_MU}, kT={DEBUG_kT}, L={L_debug:.2f}\n'
              f'Last {len(last_snaps)} snapshots (every {DEBUG_SNAP_EVERY} sweeps)',
              fontsize=11)
plt.tight_layout()
plt.show()


---
## Task 5: NVT MD — 20 random particles, no insertion/deletion

Generate 20 non-overlapping particles at random positions and orientations
(RSA), then run 2 million BAOAB Langevin MD steps (equil) + 100 000 production
steps, recording energy every 500 steps. Cluster analysis on the final frame.


In [ ]:
# ── Task 5 hyper-parameters ───────────────────────────────────────────────────
GAMMA5        = 1.0
DT5           = 1e-3
N_EQUIL5      = 1_900_000
N_PROD5       = 100_000
RECORD_EVERY5 = 500          # -> 200 energy samples in production

n_equil_chunks = N_EQUIL5  // RECORD_EVERY5   # 3800
n_prod_chunks  = N_PROD5   // RECORD_EVERY5   # 200

N5      = 20
L5      = 16.0             # packing fraction ~ 0.24 for N=20, r=1
params5 = params_debug     # E_AB=100, kT=1 -> strong bonding

print(f'Task 5 NVT MD: N={N5}, kT={DEBUG_kT}, gamma={GAMMA5}, dt={DT5}')
print(f'  box L={L5}, packing fraction {N5 * np.pi * float(CENTER_RADIUS)**2 / L5**2:.3f}')
print(f'  equil: {N_EQUIL5:,} steps ({n_equil_chunks} chunks)')
print(f'  prod : {N_PROD5:,} steps ({n_prod_chunks} chunks), record every {RECORD_EVERY5} steps')

# ── BAOAB integrator ──────────────────────────────────────────────────────────
decay5   = float(jnp.exp(-GAMMA5 * DT5))
sigma_v5 = float(jnp.sqrt(DEBUG_kT * (1.0 - decay5**2)))

@jax.jit
def _force5(q_arr):
    """All-particle forces (N,3) via grad of _total_energy_jax."""
    return -jax.grad(
        lambda qf: _total_energy_jax(qf, L5, params5)
    )(q_arr.ravel()).reshape(q_arr.shape)

def _baoab_step(carry, _):
    """One BAOAB step; carry = (q, v, key)."""
    q, v, key = carry
    F = _force5(q)
    key, sub = jax.random.split(key)
    v = v + 0.5 * DT5 * F
    q = q + 0.5 * DT5 * v
    v = decay5 * v + sigma_v5 * jax.random.normal(sub, v.shape)
    q = q + 0.5 * DT5 * v
    # Periodic wrap on positions only
    q = q.at[:, :2].set(jnp.mod(q[:, :2] + L5/2, L5) - L5/2)
    F = _force5(q)
    v = v + 0.5 * DT5 * F
    return (q, v, key), None

@jax.jit
def _run_chunk(carry):
    """Run RECORD_EVERY5 steps, return (carry, energy_at_end)."""
    carry, _ = lax.scan(_baoab_step, carry, None, length=RECORD_EVERY5)
    q, v, key = carry
    e = _total_energy_jax(q.ravel(), L5, params5)
    return carry, e

# ── Random initial config: RSA (no hard-core overlaps) ───────────────────────
rng5     = np.random.default_rng(42)
sigma_cc = 2.0 * float(CENTER_RADIUS)   # hard-core diameter
q5_np    = np.zeros((N5, 3))
placed   = 0
max_attempts = 100_000
attempts = 0
while placed < N5 and attempts < max_attempts:
    attempts += 1
    xy  = rng5.uniform(-L5/2, L5/2, size=2)
    th  = rng5.uniform(0.0, 2*np.pi)
    ok  = True
    for j in range(placed):
        dr = xy - q5_np[j, :2]
        dr -= L5 * np.round(dr / L5)   # minimum image
        if np.linalg.norm(dr) < sigma_cc:
            ok = False
            break
    if ok:
        q5_np[placed] = [xy[0], xy[1], th]
        placed += 1
if placed < N5:
    raise RuntimeError(f'RSA failed: only placed {placed}/{N5} particles after {max_attempts} attempts')
print(f'RSA placed {placed} particles in {attempts} attempts')

q5   = jnp.array(q5_np, dtype=jnp.float64)
key5 = jax.random.PRNGKey(77)
v5   = jnp.sqrt(DEBUG_kT) * jax.random.normal(key5, q5.shape)
key5, _ = jax.random.split(key5)
carry5   = (q5, v5, key5)

# ── Equilibration (1.9M steps, discard energies) ─────────────────────────────
print('Compiling + running equilibration...', flush=True)

@jax.jit
def run_equil(carry):
    return lax.fori_loop(0, n_equil_chunks,
                         lambda _, c: _run_chunk(c)[0],
                         carry)

carry5 = run_equil(carry5)
print('  equilibration done.')

# ── Production (100K steps, collect energy every 500 steps) ──────────────────
print('Running production...', flush=True)

@jax.jit
def run_prod(carry):
    return lax.scan(lambda c, _: _run_chunk(c), carry, None, length=n_prod_chunks)

carry5, e_prod5 = run_prod(carry5)
e_prod5 = np.array(e_prod5)   # (200,)
print(f'  production done.  mean E = {e_prod5.mean():.4f} +/- {e_prod5.std():.4f}')


In [ ]:

# ── Plot energy trace ─────────────────────────────────────────────────────────
step_axis = np.arange(n_prod_chunks) * RECORD_EVERY5 + N_EQUIL5

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(step_axis / 1e6, e_prod5, lw=1)
ax.set_xlabel('MD steps (x10^6)')
ax.set_ylabel('Energy')
ax.set_title(f'Task 5: production energy  (N={N5}, L={L5}, kT={DEBUG_kT}, E_AB={params5["E_AB"]})')
plt.tight_layout()
plt.show()

# ── Final configuration + cluster analysis ────────────────────────────────────
q_final = np.array(carry5[0])   # (N5, 3)

oa_f = float(OPENING_ANGLE)
rp_f = float(CENTER_RADIUS)
ps_f = float(PATCH_SIZE)

def get_patches_np(q_arr, rp, oa):
    c  = q_arr[:, :2]
    th = q_arr[:, 2]
    pA = c + rp * np.stack([np.cos(th),      np.sin(th)],      axis=-1)
    pB = c + rp * np.stack([np.cos(th + oa), np.sin(th + oa)], axis=-1)
    return pA, pB

def min_img(dr, L):
    return dr - L * np.round(dr / L)

def find_bonds(q_arr, L, rp, oa, patch_sigma):
    N = len(q_arr)
    pA, pB = get_patches_np(q_arr, rp, oa)
    bonds = []
    for i in range(N):
        for j in range(i+1, N):
            dists = [
                np.linalg.norm(min_img(pA[i] - pA[j], L)),
                np.linalg.norm(min_img(pA[i] - pB[j], L)),
                np.linalg.norm(min_img(pB[i] - pA[j], L)),
                np.linalg.norm(min_img(pB[i] - pB[j], L)),
            ]
            if min(dists) < patch_sigma:
                bonds.append((i, j))
    return bonds

def connected_components(N, bonds):
    parent = list(range(N))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i, j in bonds:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[ri] = rj
    roots = {find(i) for i in range(N)}
    clusters = {r: [i for i in range(N) if find(i) == r] for r in roots}
    return list(clusters.values())

bonds5    = find_bonds(q_final, L5, rp_f, oa_f, ps_f)
clusters5 = connected_components(N5, bonds5)
clusters5.sort(key=len, reverse=True)

print(f'\nFinal state: N={N5}, {len(bonds5)} bonds, {len(clusters5)} cluster(s)')
for k, cl in enumerate(clusters5):
    print(f'  Cluster {k}: size {len(cl)}, particles {cl}')

# ── Draw final config coloured by cluster ─────────────────────────────────────
cmap5 = plt.cm.get_cmap('tab10', max(len(clusters5), 1))
node_color = np.zeros(N5, dtype=int)
for k, cl in enumerate(clusters5):
    for idx in cl:
        node_color[idx] = k

pA_f, pB_f = get_patches_np(q_final, rp_f, oa_f)

fig2, ax2 = plt.subplots(figsize=(6, 6))
ax2.set_xlim(-L5/2, L5/2); ax2.set_ylim(-L5/2, L5/2)
ax2.set_aspect('equal')
rect = plt.Rectangle((-L5/2, -L5/2), L5, L5,
                      fill=False, edgecolor='dimgray', lw=1.5)
ax2.add_patch(rect)

# Bonds (minimum-image)
for i, j in bonds5:
    ci = q_final[i, :2]
    cj = q_final[j, :2]
    dr = min_img(cj - ci, L5)
    ax2.plot([ci[0], ci[0]+dr[0]], [ci[1], ci[1]+dr[1]],
             'k-', lw=2.5, alpha=0.4, zorder=1)

# Particles
for idx in range(N5):
    xi, yi, thi = q_final[idx]
    col = cmap5(node_color[idx])
    circ = Circle((xi, yi), rp_f, facecolor=col, edgecolor='k',
                   alpha=0.45, lw=1.0, zorder=2)
    ax2.add_patch(circ)
    ax2.annotate('', xy=pA_f[idx], xytext=(xi, yi),
                 arrowprops=dict(arrowstyle='->', color='crimson',   lw=1.8), zorder=3)
    ax2.annotate('', xy=pB_f[idx], xytext=(xi, yi),
                 arrowprops=dict(arrowstyle='->', color='limegreen', lw=1.8), zorder=3)
    ax2.text(xi+0.05, yi+0.05, str(idx), fontsize=9, zorder=4)

cluster_legend = [
    mpatches.Patch(color=cmap5(k),
                   label=f'cluster {k}  (size {len(cl)})')
    for k, cl in enumerate(clusters5)
]
ax2.legend(handles=cluster_legend, loc='upper right', fontsize=9)
ax2.set_title(
    f'Task 5 final config -- {len(clusters5)} cluster(s), {len(bonds5)} bond(s)\n'
    f'(E_AB={DEBUG_E_AB}, kT={DEBUG_kT}, N={N5})'
)
plt.tight_layout()
plt.show()
